In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# 1달치 목업 데이터 로드
df = pd.read_csv('output.csv')
df.head()

,avg_dwell_time,core_customer_age,core_customer_gender,just_left_count,max_empty_table_time,max_response_wait_time,peak_time,temperature,total_count,captured_at,created_at,end_at,id,weather
0,70,30,1,0,18,4,19,19.4,52,2026-04-30 11:00:00.000000,2026-05-30 22:01:57.212084,2026-04-30 22:00:00.000000,1,SUNNY
1,67,30,1,1,12,7,18,19.8,35,2026-05-01 11:00:00.000000,2026-05-30 22:01:57.570298,2026-05-01 22:00:00.000000,2,SUNNY
2,68,30,1,1,32,4,19,18.5,59,2026-05-02 11:00:00.000000,2026-05-30 22:01:57.842495,2026-05-02 22:00:00.000000,3,CLOUDY
3,68,30,1,0,21,5,19,13.2,64,2026-05-03 11:00:00.000000,2026-05-30 22:02:01.373380,2026-05-03 22:00:00.000000,4,RAINY
4,72,30,1,3,19,4,18,16.1,37,2026-05-04 11:00:00.000000,2026-05-30 22:02:01.782206,2026-05-04 22:00:00.000000,5,CLOUDY


In [3]:
# [피처 엔지니어링 1단계] captured_at 날짜 파싱 및 요일 번호 생성
df['captured_at'] = pd.to_datetime(df['captured_at'])
df['day_of_week'] = df['captured_at'].dt.dayofweek
df[['captured_at', 'day_of_week']].head()

,captured_at,day_of_week
0,2026-04-30 11:00:00,3
1,2026-05-01 11:00:00,4
2,2026-05-02 11:00:00,5
3,2026-05-03 11:00:00,6
4,2026-05-04 11:00:00,0


In [4]:
# [피처 엔지니어링 2단계] 주말 여부(is_weekend) 바이너리 변수 생성
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df[['captured_at', 'day_of_week', 'is_weekend']].head()

,captured_at,day_of_week,is_weekend
0,2026-04-30 11:00:00,3,0
1,2026-05-01 11:00:00,4,0
2,2026-05-02 11:00:00,5,1
3,2026-05-03 11:00:00,6,1
4,2026-05-04 11:00:00,0,0


In [5]:
# [피처 엔지니어링 3단계] 어제 방문객 수(prev_day_count) 시계열 지연 생성 및 결측치 제거
df['prev_day_count'] = df['total_count'].shift(1)
df_clean = df.dropna().reset_index(drop=True)
df_clean[['captured_at', 'is_weekend', 'prev_day_count', 'total_count']].head()

,captured_at,is_weekend,prev_day_count,total_count
0,2026-05-01 11:00:00,0,52.0,35
1,2026-05-02 11:00:00,1,35.0,59
2,2026-05-03 11:00:00,1,59.0,64
3,2026-05-04 11:00:00,0,64.0,37
4,2026-05-05 11:00:00,0,37.0,38


In [6]:
# [피처 엔지니어링 4단계] 범주형 기상(weather) 데이터 원-핫 인코딩 적용
df_encoded = pd.get_dummies(df_clean, columns=['weather'], drop_first=True)
df_encoded.head()

,avg_dwell_time,core_customer_age,core_customer_gender,just_left_count,max_empty_table_time,max_response_wait_time,peak_time,temperature,total_count,captured_at,created_at,end_at,id,day_of_week,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,67,30,1,1,12,7,18,19.8,35,2026-05-01 11:00:00,2026-05-30 22:01:57.570298,2026-05-01 22:00:00.000000,2,4,0,52.0,False,True
1,68,30,1,1,32,4,19,18.5,59,2026-05-02 11:00:00,2026-05-30 22:01:57.842495,2026-05-02 22:00:00.000000,3,5,1,35.0,False,False
2,68,30,1,0,21,5,19,13.2,64,2026-05-03 11:00:00,2026-05-30 22:02:01.373380,2026-05-03 22:00:00.000000,4,6,1,59.0,True,False
3,72,30,1,3,19,4,18,16.1,37,2026-05-04 11:00:00,2026-05-30 22:02:01.782206,2026-05-04 22:00:00.000000,5,0,0,64.0,False,False
4,67,30,1,1,32,4,18,16.9,38,2026-05-05 11:00:00,2026-05-30 22:02:02.046001,2026-05-05 22:00:00.000000,6,1,0,37.0,False,True


In [7]:
# [피처 엔지니어링 5단계] 미래 정보 배제(Data Leakage 차단) 및 최종 피처 선택
features = ['temperature', 'is_weekend', 'prev_day_count']
features += [col for col in df_encoded.columns if col.startswith('weather_')]

X = df_encoded[features]
y = df_encoded['total_count']
X.head()

,temperature,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,19.8,0,52.0,False,True
1,18.5,1,35.0,False,False
2,13.2,1,59.0,True,False
3,16.1,0,64.0,False,False
4,16.9,0,37.0,False,True


In [8]:
# 릿지 회귀 규제 작동을 위한 StandardScaler 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 학습용 80%, 검증용 20% 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 주피터 노트북에 예쁜 판다스 표로 스케일링된 첫 5개 행 상태 보여주기
pd.DataFrame(X_scaled, columns=features).head()

,temperature,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,-0.372387,-0.617213,0.366773,-0.456435,1.378405
1,-0.705064,1.620185,-1.178689,-0.456435,-0.725476
2,-2.061363,1.620185,1.003140,2.190890,-0.725476
3,-1.319237,-0.617213,1.457687,-0.456435,-0.725476
4,-1.114513,-0.617213,-0.996870,-0.456435,1.378405


In [9]:
# 그리드서치를 위한 하이퍼파라미터 후보군 선언
param_grid = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
}

# 5-Fold 교차 검증 기반 GridSearchCV 릿지 학습 수행
grid_search = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5
)
grid_search.fit(X_train, y_train)

# 최적 모델 획득
best_model = grid_search.best_estimator_
best_alpha = grid_search.best_params_['alpha']

# 검증 평가 및 성능 출력
y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"최적 규제 하이퍼파라미터 (Best alpha): {best_alpha}")
print(f"평균 절대 오차 (MAE): {mae:.2f} 명")
print(f"결정계수 (R² Score): {r2:.4f}\n")

# 최적 모델 가중치 결과를 판다스 데이터프레임 표(HTML Table) 형식으로 직접 리턴
coef_df = pd.DataFrame({
    '최적 입력 피처 (Best Feature)': features,
    '최적 영향도 가중치 (Coefficients)': best_model.coef_
})
coef_df

최적 규제 하이퍼파라미터 (Best alpha): 2.0
평균 절대 오차 (MAE): 5.96 명
결정계수 (R² Score): 0.6552



,최적 입력 피처 (Best Feature),최적 영향도 가중치 (Coefficients)
0,temperature,0.804538
1,is_weekend,7.816925
2,prev_day_count,-0.618817
3,weather_RAINY,1.306211
4,weather_SUNNY,-0.169930


In [10]:
def predict_tomorrow(tomorrow_temp, tomorrow_weather, today_count, model, scaler, feature_columns):
    # 내일 날짜 요일 기준 계산 (오늘 + 1일)
    tomorrow_date = pd.Timestamp.now() + pd.Timedelta(days=1)
    day_of_week = tomorrow_date.dayofweek
    is_weekend = 1 if day_of_week >= 5 else 0
    
    input_data = {
        'temperature': tomorrow_temp,
        'is_weekend': is_weekend,
        'prev_day_count': today_count
    }
    
    # 원핫 인코딩 날씨 매핑
    for col in feature_columns:
        if col.startswith('weather_'):
            weather_type = col.replace('weather_', '')
            input_data[col] = 1 if tomorrow_weather.upper() == weather_type.upper() else 0
            
    input_df = pd.DataFrame([input_data])[feature_columns]
    input_scaled = scaler.transform(input_df)
    
    # 예측 수행
    pred_raw = model.predict(input_scaled)[0]
    final_pred = max(0, int(round(pred_raw)))
    
    kor_days = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    
    # 최종 예측 보고서를 판다스 표(HTML Table) 형식으로 직접 리턴
    result_df = pd.DataFrame({
        '예측 항목': ['예측 타겟 날짜', '최적 alpha 값', '내일 기온 예보', '내일 기상 상태', '오늘 최종 방문객', '🔮 내일 예상 유입 고객수'],
        '예측 결과값': [
            f"{tomorrow_date.strftime('%Y-%m-%d')} ({kor_days[day_of_week]})",
            f"{best_alpha}",
            f"{tomorrow_temp} ℃",
            tomorrow_weather.upper(),
            f"{today_count} 명",
            f"{final_pred} 명"
        ]
    })
    return result_df

# 가상 상황 시뮬레이션 가동 및 표 리턴
predict_tomorrow(
    tomorrow_temp=21.5, 
    tomorrow_weather='RAINY', 
    today_count=42, 
    model=best_model, 
    scaler=scaler, 
    feature_columns=features
)

,예측 항목,예측 결과값
0,예측 타겟 날짜,2026-06-01 (월요일)
1,최적 alpha 값,2.0
2,내일 기온 예보,21.5 ℃
3,내일 기상 상태,RAINY
4,오늘 최종 방문객,42 명
5,🔮 내일 예상 유입 고객수,46 명
